# AGMS-Net Phase 2: Regime-Aware Gating (v4 — Multiplicative, FIXED metrics)

> **CRITICAL FIX:** `metrics.npy` stores `[MAE, MSE, RMSE, ...]` (MAE first).
> Previous notebook versions read `m[0]` as MSE — **wrong**. This version reads them correctly.

**Architecture:** `RegimeDetector` extracts 3 scale-invariant features (norm_trend, kurtosis, autocorr) from pre-RevIN input → MLP (3→32→16) → `regime_proj` (16→96) → multiplicative gate `seasonal_init *= (1 + scale·tanh(proj))`.

**Runtime:** GPU required (Runtime > Change runtime type > T4 GPU)

## 1. Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Clone Repository

In [ ]:
import os
%cd /content
!rm -rf /content/Workload_Prediction_MTP
!git clone https://github.com/patelkanak23/Workload_Prediction_MTP.git /content/Workload_Prediction_MTP
%cd /content/Workload_Prediction_MTP/MSCNet_baseline
print('run.py exists:', os.path.exists('run.py'))

with open('models/MSCNet.py') as f:
    content = f.read()
    assert 'RegimeDetector' in content, 'ERROR: RegimeDetector not found!'
    assert 'regime_gate' in content, 'ERROR: regime_gate not found — not v4!'
    print('Phase 2 v4 confirmed (multiplicative regime gating)')

## 3. Install Dependencies

In [ ]:
!pip install -q einops ptflops torchinfo reformer-pytorch PyWavelets sktime patool matplotlib tqdm

## 4. Upload Dataset

In [ ]:
import os, pandas as pd
%cd /content
os.makedirs('dataset', exist_ok=True)
if not os.path.exists('dataset/alibaba.csv'):
    from google.colab import files
    print('Upload alibaba.csv (96 machines, ~11MB):')
    uploaded = files.upload()
    for f in uploaded: os.rename(f, 'dataset/alibaba.csv')
df = pd.read_csv('dataset/alibaba.csv')
n_features = len(df.columns) - 1
print(f'Dataset: {df.shape}, Features: {n_features}')
print(f'Time: {df["date"].iloc[0]} to {df["date"].iloc[-1]}')

## 5. Setup Paths & Symlinks

In [ ]:
import os
%cd /content
DATA_DIR  = '/content/dataset'
WORK_DIR  = '/content/Workload_Prediction_MTP/MSCNet_baseline'
os.makedirs(f'{WORK_DIR}/dataset', exist_ok=True)
if not os.path.exists(f'{WORK_DIR}/dataset/alibaba.csv'):
    os.symlink(f'{DATA_DIR}/alibaba.csv', f'{WORK_DIR}/dataset/alibaba.csv')
print('Symlink created:', os.path.exists(f'{WORK_DIR}/dataset/alibaba.csv'))
%cd {WORK_DIR}
print('Working directory:', os.getcwd())

## 6. Train All Horizons (H=24, 48, 72, 96)

~8-12 min each on T4.

In [ ]:
import os, subprocess, sys

HORIZONS = [24, 48, 72, 96]
N_FEATURES = 96
SEQ_LEN = 96

for h in HORIZONS:
    print(f'\n{"="*80}')
    print(f'  Training H={h}')
    print(f'{"="*80}\n')

    cmd = [
        sys.executable, 'run.py',
        '--task_name', 'long_term_forecast',
        '--is_training', '1',
        '--root_path', './dataset/',
        '--data_path', 'alibaba.csv',
        '--model_id', f'alibaba_{SEQ_LEN}_{h}',
        '--model', 'MSCNet',
        '--data', 'custom',
        '--features', 'M',
        '--seq_len', str(SEQ_LEN),
        '--label_len', str(SEQ_LEN),
        '--pred_len', str(h),
        '--e_layers', '3',
        '--d_layers', '1',
        '--factor', '3',
        '--enc_in', str(N_FEATURES),
        '--dec_in', str(N_FEATURES),
        '--c_out', str(N_FEATURES),
        '--d_model', '128',
        '--d_ff', '2048',
        '--n_heads', '4',
        '--des', 'Exp',
        '--itr', '1',
        '--batch_size', '32',
        '--learning_rate', '0.0001',
        '--train_epochs', '100',
        '--patience', '3',
        '--channel', str(N_FEATURES),
        '--seed', '2022',
    ]

    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        print(f'ERROR: H={h} failed with return code {result.returncode}')

## 7. Collect & Compare Results

> **FIXED:** `metrics.npy` stores `[MAE, MSE, RMSE, MAPE, MSPE, ms]`.
> `m[0]` = MAE, `m[1]` = MSE, `m[2]` = RMSE.

In [ ]:
import os, re, numpy as np

phase0 = {24: (0.4630, 0.4927), 48: (0.5638, 0.5438), 72: (0.6447, 0.5794), 96: (0.7213, 0.6144)}
paper  = {24: 0.5687, 48: 0.6274, 72: 0.6622, 96: 0.6893}

results = {}
print('--- Raw Results Log ---')
print('NOTE: metrics.npy order is [MAE, MSE, RMSE, MAPE, MSPE, ms]')
print()
for root, dirs, files in os.walk('results'):
    for f in files:
        if f == 'metrics.npy':
            path = os.path.join(root, f)
            m = np.load(path)
            folder = os.path.basename(root)
            # CORRECT ORDER: m[0]=MAE, m[1]=MSE, m[2]=RMSE
            actual_mae = m[0]
            actual_mse = m[1]
            actual_rmse = m[2]
            print(f'{folder}')
            print(f'mse:{actual_mse}, mae:{actual_mae}, rmse:{actual_rmse}')
            # Verify: sqrt(MSE) should equal RMSE
            check = abs(np.sqrt(actual_mse) - actual_rmse) < 0.001
            print(f'  sqrt(MSE)={np.sqrt(actual_mse):.4f} vs RMSE={actual_rmse:.4f} -> {"OK" if check else "MISMATCH!"}')
            match = re.search(r'alibaba_96_(\d+)_', folder)
            if match:
                h = int(match.group(1))
                results[h] = (float(actual_mse), float(actual_mae))
            print()

print()
print('=' * 110)
print('AGMS-Net Phase 2 v4 vs Phase 0 Baseline vs Paper (Alibaba, L=96) [CORRECTED METRICS]')
print('=' * 110)
hdr = f'{"H":>4} | {"P2v4 MSE":>10} | {"Phase0 MSE":>10} | {"vs P0":>7} | {"Paper MSE":>10} | {"P2v4 MAE":>10} | {"Phase0 MAE":>10} | {"vs P0":>7}'
print(hdr)
print('-' * 110)
for h in [24, 48, 72, 96]:
    if h in results:
        p_mse, p_mae = results[h]
        p0_mse, p0_mae = phase0[h]
        d_mse = (p_mse - p0_mse) / p0_mse * 100
        d_mae = (p_mae - p0_mae) / p0_mae * 100
        print(f'{h:4d} | {p_mse:10.4f} | {p0_mse:10.4f} | {d_mse:+6.1f}% | {paper[h]:10.4f} | {p_mae:10.4f} | {p0_mae:10.4f} | {d_mae:+6.1f}%')
print('=' * 110)

## 8. Regime Scale & Gate Analysis

In [ ]:
import torch, os, glob

checkpoint_dirs = sorted(glob.glob('checkpoints/long_term_forecast_alibaba_*'))
print(f'Found {len(checkpoint_dirs)} checkpoint(s)')

for ckpt_dir in checkpoint_dirs:
    ckpt_path = os.path.join(ckpt_dir, 'checkpoint.pth')
    if not os.path.exists(ckpt_path): continue
    print(f'\n--- {ckpt_path} ---')
    state = torch.load(ckpt_path, map_location='cpu')

    if 'regime_scale' in state:
        rs = state['regime_scale'].item()
        print(f'  regime_scale = {rs:.6f}  (init was 0.01)')

    regime_params = {k: v for k, v in state.items() if 'regime' in k}
    for k, v in regime_params.items():
        if k == 'regime_scale': continue
        print(f'  {k}: mean={v.mean():.4f}, std={v.std():.4f}')
    total = sum(v.numel() for v in state.values())
    rp = sum(v.numel() for v in regime_params.values())
    print(f'  Regime params: {rp:,} / {total:,} ({rp/total*100:.2f}%)')

## 9. Model Complexity

In [ ]:
import torch, sys, os
os.chdir('/content/Workload_Prediction_MTP/MSCNet_baseline')
sys.path.insert(0, '.')
from ptflops import get_model_complexity_info
from types import SimpleNamespace
configs = SimpleNamespace(
    task_name='long_term_forecast', seq_len=96, pred_len=24,
    enc_in=96, dec_in=96, c_out=96, d_model=128, d_ff=2048,
    n_heads=4, e_layers=3, d_layers=1, factor=3, dropout=0.05,
    output_attention=False, activation='gelu', moving_avg=25, channel=96
)
from models.MSCNet import Model
model = Model(configs)
macs, params = get_model_complexity_info(model, (96, 96), as_strings=True, print_per_layer_stat=False, verbose=False)
print(f'MACs: {macs}')
print(f'Parameters: {params}')